# 77館冰水主機監控資料 + 竹東站氣象 逐時整理

將監控資料與竹東站逐時氣象整理成每小時一列，輸出 Excel。

- **監控**：`77館材化所CH1`、`77館材化所CH2` 的 5 個項目（功率、流量、進／回水溫度、冷凍噸），每分鐘 → 每小時平均
- **氣象**：竹東站（C0D560）`weather_history` 的 temperature、humidity、pressure、wind_speed；缺資料的小時以內插補值
- **欄位名稱**：監控為「CH1_監測項目」，氣象為 DB 欄位名
- **時間軸**：台北時間，每小時連續，沒有資料的小時留空
- **輸出**：`analysis/output/chiller_weather_<測站代號>_<起始日>_<結束日>.xlsx`（工作表 `hourly`、`說明`）

## 1. 環境設定

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from openpyxl.utils import get_column_letter

# 切到專案根目錄，才 import 得到 db、data、model
if Path.cwd().name == "analysis":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import settings
from data import weather
from data.sensor import get_metric_data  # 抓一個監測項目，index 為台灣時間

TZ = settings.settings.timezone
STATION_ID = settings.settings.cwa_realtime_station_id
OUTPUT_DIR = Path("analysis/output")

## 2. 監控資料（CH-1 / CH-2，每小時平均）

In [2]:
METRIC_IDS = ["77館材化所CH1", "77館材化所CH2"]
METRIC_KEYS = {  # 監測項目: 單位
    "電錶即時功率(通訊)": "kW",
    "冰水流量": "LPM",
    "冰水流量進水溫度": "°C",
    "冰水流量回水溫度": "°C",
    "冰水流量冷凍噸": "RT",
}


# 點位簡稱：去掉「77館材化所」
def short_name(metric_id: str) -> str:
    return metric_id.removeprefix("77館材化所")


# 一台主機的 5 個項目取每小時平均，欄名為「CH1_監測項目」
async def load_chiller_hourly(metric_id: str) -> pd.DataFrame:
    data = {}
    for key in METRIC_KEYS:
        data[f"{short_name(metric_id)}_{key}"] = await get_metric_data(metric_id, key)
    return pd.DataFrame(data).sort_index().resample("1h").mean()


chiller_frames = []
for metric_id in METRIC_IDS:
    chiller_frames.append(await load_chiller_hourly(metric_id))
df_sensor = pd.concat(chiller_frames, axis=1)
print(
    "監控資料：",
    len(df_sensor),
    "小時，",
    df_sensor.index.min(),
    "~",
    df_sensor.index.max(),
)

監控資料： 623 小時， 2026-09-17 13:00:00+08:00 ~ 2026-10-13 11:00:00+08:00


## 3. 氣象資料（竹東站逐時觀測）

In [3]:
WEATHER_UNITS = {  # 輸出的氣象欄位: 單位
    "temperature": "°C",
    "humidity": "%",
    "pressure": "hPa",
    "wind_speed": "m/s",
}

start = int(df_sensor.index.min().timestamp())
# load_history 不含 end，多抓一小時
end = int(df_sensor.index.max().timestamp()) + 3_600

weather_history = await weather.load_history(start, end)
df_weather = weather.to_hourly(weather_history)[list(WEATHER_UNITS)]
print(
    "氣象資料（測站",
    STATION_ID,
    "）：",
    len(df_weather),
    "小時，",
    df_weather.index.min(),
    "~",
    df_weather.index.max(),
)

氣象資料（測站 C0D560 ）： 623 小時， 2026-09-17 13:00:00+08:00 ~ 2026-10-13 11:00:00+08:00


## 4. 合併成每小時一列

In [4]:
# 補齊每個小時，缺資料留空
hours = pd.date_range(
    df_sensor.index.min(),
    df_sensor.index.max(),
    freq="1h",
    tz=TZ,
)

hourly_data = pd.concat([df_sensor, df_weather], axis=1).reindex(hours)
hourly_data.index = hourly_data.index.tz_localize(None)  # Excel 不支援帶時區的時間
hourly_data.index.name = "時間(台北)"

print(
    "總列數：",
    len(hourly_data),
    "，時間範圍：",
    hourly_data.index.min(),
    "~",
    hourly_data.index.max(),
)
print()
print("各欄有資料的小時數：")
print(hourly_data.notna().sum().to_string())

總列數： 623 ，時間範圍： 2026-09-17 13:00:00 ~ 2026-10-13 11:00:00

各欄有資料的小時數：
CH1_電錶即時功率(通訊)    623
CH1_冰水流量          623
CH1_冰水流量進水溫度      623
CH1_冰水流量回水溫度      623
CH1_冰水流量冷凍噸       623
CH2_電錶即時功率(通訊)    623
CH2_冰水流量          623
CH2_冰水流量進水溫度      623
CH2_冰水流量回水溫度      623
CH2_冰水流量冷凍噸       623
temperature       623
humidity          623
pressure          623
wind_speed        623


In [5]:
hourly_data.head()

,CH1_電錶即時功率(通訊),CH1_冰水流量,CH1_冰水流量進水溫度,CH1_冰水流量回水溫度,CH1_冰水流量冷凍噸,CH2_電錶即時功率(通訊),CH2_冰水流量,CH2_冰水流量進水溫度,CH2_冰水流量回水溫度,CH2_冰水流量冷凍噸,temperature,humidity,pressure,wind_speed
時間(台北),,,,,,,,,,,,,,
2026-09-17 13:00:00,10.783784,1.945946,23.586486,23.900000,0.0,197.864865,3749.459459,8.572973,13.035135,333.054054,29.9,60.0,996.0,1.1
2026-09-17 14:00:00,11.920000,2.000000,23.580000,23.918000,0.0,216.300000,3978.280000,8.634000,13.138000,353.580000,33.0,51.0,995.5,1.0
2026-09-17 15:00:00,12.700000,2.000000,23.613333,24.050000,0.0,212.916667,4210.633333,8.615000,12.768333,346.750000,32.0,56.0,995.3,1.6
2026-09-17 16:00:00,10.916667,2.000000,23.593333,24.100000,0.0,195.583333,4209.533333,8.643333,12.471667,319.483333,30.8,61.0,995.4,1.5
2026-09-17 17:00:00,3.266667,2.000000,23.651667,24.248333,0.0,48.733333,1510.883333,11.471667,12.981667,94.883333,29.2,69.0,995.7,1.3


## 5. 欄位說明

In [6]:
column_rows = []
for metric_id in METRIC_IDS:
    for key, unit in METRIC_KEYS.items():
        column_rows.append(
            {
                "欄位": f"{short_name(metric_id)}_{key}",
                "來源": "sensor_data",
                "點位": short_name(metric_id),
                "監測項目": key,
                "單位": unit,
                "備註": "每分鐘資料取每小時平均",
            }
        )
for column, unit in WEATHER_UNITS.items():
    column_rows.append(
        {
            "欄位": column,
            "來源": "weather_history",
            "點位": f"測站 {STATION_ID}",
            "監測項目": column,
            "單位": unit,
            "備註": "逐時觀測，缺資料的小時內插補值",
        }
    )

column_dictionary = pd.DataFrame(column_rows)
column_dictionary

,欄位,來源,點位,監測項目,單位,備註
0,CH1_電錶即時功率(通訊),sensor_data,CH1,電錶即時功率(通訊),kW,每分鐘資料取每小時平均
1,CH1_冰水流量,sensor_data,CH1,冰水流量,LPM,每分鐘資料取每小時平均
2,CH1_冰水流量進水溫度,sensor_data,CH1,冰水流量進水溫度,°C,每分鐘資料取每小時平均
3,CH1_冰水流量回水溫度,sensor_data,CH1,冰水流量回水溫度,°C,每分鐘資料取每小時平均
4,CH1_冰水流量冷凍噸,sensor_data,CH1,冰水流量冷凍噸,RT,每分鐘資料取每小時平均
5,CH2_電錶即時功率(通訊),sensor_data,CH2,電錶即時功率(通訊),kW,每分鐘資料取每小時平均
6,CH2_冰水流量,sensor_data,CH2,冰水流量,LPM,每分鐘資料取每小時平均
7,CH2_冰水流量進水溫度,sensor_data,CH2,冰水流量進水溫度,°C,每分鐘資料取每小時平均
8,CH2_冰水流量回水溫度,sensor_data,CH2,冰水流量回水溫度,°C,每分鐘資料取每小時平均
9,CH2_冰水流量冷凍噸,sensor_data,CH2,冰水流量冷凍噸,RT,每分鐘資料取每小時平均


## 6. 輸出 Excel

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
first_hour = hourly_data.index.min()
last_hour = hourly_data.index.max()
file_name = f"chiller_weather_{STATION_ID}_{first_hour:%Y%m%d}_{last_hour:%Y%m%d}.xlsx"
output_path = OUTPUT_DIR / file_name

with pd.ExcelWriter(
    output_path,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd hh:mm",
) as writer:
    hourly_data.round(3).to_excel(writer, sheet_name="hourly")
    column_dictionary.to_excel(writer, sheet_name="說明", index=False)

    # 欄寬：時間欄 18、資料欄 24；凍結首列、首欄
    worksheet = writer.sheets["hourly"]
    worksheet.freeze_panes = "B2"
    worksheet.column_dimensions["A"].width = 18
    for i in range(2, worksheet.max_column + 1):
        worksheet.column_dimensions[get_column_letter(i)].width = 24

    worksheet = writer.sheets["說明"]
    for column_letter, width in zip("ABCDEF", [26, 16, 10, 22, 8, 24]):
        worksheet.column_dimensions[column_letter].width = width

print("已輸出：", output_path.resolve())

已輸出： C:\Users\hsinjou.chang\Documents\chiller_77\analysis\output\chiller_weather_C0D560_20260917_20261013.xlsx
